# Robustness Checks
Compact experiments to test stability of the pipeline: target horizon, macro overlay, and feature selection.


## Setup


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from notebooks.params import OUT_DIR
from src.modeling.baseline import BaselineConfig, add_momentum_score
from src.modeling.backtest import BacktestConfig, backtest_from_scores, top_k_hit_rate
from src.modeling.metrics import mean_monthly_spearman_ic
from src.modeling.models import ModelIO, build_ridge_pipeline, build_tree_model, fit_predict_oos_scores
from src.modeling.splits import WalkForwardConfig, generate_expanding_walk_forward_splits

STAGES_DIR = Path(OUT_DIR) / "stages"
MODEL_PATH = STAGES_DIR / "panel_model_ready.parquet"


## Load model‑ready panel


In [ ]:
df_model = pd.read_parquet(MODEL_PATH)
print("Rows:", len(df_model))


## Common config


In [ ]:
split_cfg = WalkForwardConfig(train_years=10, test_months=12, embargo_months=3)
splits = list(generate_expanding_walk_forward_splits(df_model, split_cfg))

n_tickers = df_model['ticker'].nunique()
if n_tickers >= 40:
    top_k = 5
elif n_tickers >= 25:
    top_k = 4
else:
    top_k = 3

bt_cfg = BacktestConfig(top_k=top_k, overlay_enabled=True, stress_threshold=0.5, risk_off_exposure=0.6)

features_all = [c for c in df_model.columns if c not in {
    'date', 'ticker',
    'target_3m', 'target_1m',
    'fwd_ret_1m', 'fwd_ret_3m', 'fwd_ret_6m',
    'macro_regime', 'stress_index', 'slope_10y2y',
}]


## Robustness A: Target 3M vs 1M
We compare model performance when training on 3‑month vs 1‑month targets.
This checks whether the signal is robust to horizon choice.


In [ ]:
def run_model(target_col, score_col):
    io = ModelIO(target_col=target_col)
    model = build_ridge_pipeline(alpha=1.0)
    oos, _ = fit_predict_oos_scores(splits, features_all, model, io, score_col)
    return oos

base_cfg = BaselineConfig(momentum_col='mom12_pr', score_col='baseline_mom')
df_base = add_momentum_score(df_model, base_cfg)

# target_3m
oos_3m = run_model('target_3m', 'score_ridge_3m')
oos_3m = oos_3m.merge(df_base[['date','ticker','baseline_mom']], on=['date','ticker'], how='left')
summary_3m, _ = backtest_from_scores(oos_3m, {'baseline_mom': 'baseline_mom', 'ridge_3m': 'score_ridge_3m'}, bt_cfg)

# target_1m
oos_1m = run_model('target_1m', 'score_ridge_1m')
oos_1m = oos_1m.merge(df_base[['date','ticker','baseline_mom']], on=['date','ticker'], how='left')
summary_1m, _ = backtest_from_scores(oos_1m, {'baseline_mom': 'baseline_mom', 'ridge_1m': 'score_ridge_1m'}, bt_cfg)

print('Target 3M')
display(summary_3m)
print('Target 1M')
display(summary_1m)


## Robustness B: Macro overlay on/off
We keep the model unchanged and toggle the risk‑off overlay to see its impact on portfolio performance.


In [ ]:
io = ModelIO()
model = build_ridge_pipeline(alpha=1.0)
oos, _ = fit_predict_oos_scores(splits, features_all, model, io, 'score_ridge')
oos = oos.merge(df_base[['date','ticker','baseline_mom']], on=['date','ticker'], how='left')

bt_on = BacktestConfig(top_k=top_k, overlay_enabled=True, stress_threshold=0.5, risk_off_exposure=0.6)
bt_off = BacktestConfig(top_k=top_k, overlay_enabled=False)

summary_on, _ = backtest_from_scores(oos, {'ridge': 'score_ridge'}, bt_on)
summary_off, _ = backtest_from_scores(oos, {'ridge': 'score_ridge'}, bt_off)

print('Overlay ON')
display(summary_on)
print('Overlay OFF')
display(summary_off)


## Robustness C: Feature selection (funds‑only vs techs‑only)
We evaluate fundamentals‑only, technicals‑only, and the full feature set to assess feature group contribution.


In [ ]:
fund_cols = [c for c in features_all if c.endswith('_pr') and any(k in c for k in ['roe','roa','margin','growth','debt','coverage','turnover','fcf','delta'])]
tech_cols = [c for c in features_all if c in ['mom12_pr','mom6_pr','mom3_pr','trend_ratio_pr','vol_pr','vol_ratio_pr']]

io = ModelIO()
model = build_ridge_pipeline(alpha=1.0)

# funds only
oos_fund, _ = fit_predict_oos_scores(splits, fund_cols, model, io, 'score_fund')
summary_fund, _ = backtest_from_scores(oos_fund, {'funds_only': 'score_fund'}, bt_cfg)

# tech only
oos_tech, _ = fit_predict_oos_scores(splits, tech_cols, model, io, 'score_tech')
summary_tech, _ = backtest_from_scores(oos_tech, {'tech_only': 'score_tech'}, bt_cfg)

print('Funds only')
display(summary_fund)
print('Tech only')
display(summary_tech)

# both (funds + tech)
oos_all, _ = fit_predict_oos_scores(splits, features_all, model, io, 'score_all')
summary_all, _ = backtest_from_scores(oos_all, {'both': 'score_all'}, bt_cfg)

print('Both (funds + tech)')
display(summary_all)


## Robustness D: Feature importance (Ridge vs HGB)
We show the most influential features for the linear model (absolute coefficients) and the tree model (feature importances).
This is a compact interpretability check, not an exhaustive analysis.


In [ ]:
import matplotlib.pyplot as plt

X = df_model[features_all]
y = df_model['target_3m']

ridge = build_ridge_pipeline(alpha=1.0)
ridge.fit(X, y)
ridge_coef = pd.Series(ridge.named_steps['reg'].coef_, index=features_all)
ridge_coef = ridge_coef.reindex(ridge_coef.abs().sort_values(ascending=False).index)
display(ridge_coef.head(15))

hgb = build_tree_model()
hgb.fit(X, y)
hgb_imp = pd.Series(hgb.feature_importances_, index=features_all).sort_values(ascending=False)
display(hgb_imp.head(15))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ridge_coef.head(10)[::-1].plot(kind='barh', ax=axes[0], title='Ridge |coef| (top 10)')
hgb_imp.head(10)[::-1].plot(kind='barh', ax=axes[1], title='HGB importance (top 10)')
plt.tight_layout()


## Robustness E: Rolling 12M IC (Ridge, target 3M)
We track the stability of the Information Coefficient over time using a 12‑month rolling window.
Stable, positive IC suggests the signal is persistent; high volatility suggests fragility.


In [ ]:
def monthly_ic(df, score_col, ret_col):
    def _ic(x):
        if x[score_col].nunique() < 2 or x[ret_col].nunique() < 2:
            return np.nan
        return x[[score_col, ret_col]].corr(method='spearman').iloc[0, 1]
    return df.groupby('date').apply(_ic)

ic_series = monthly_ic(oos_3m, 'score_ridge_3m', 'fwd_ret_3m')
ic_roll = ic_series.rolling(12, min_periods=6).mean()

plt.figure(figsize=(10, 4))
ic_roll.plot(title='Rolling 12M IC (Ridge, target 3M)')
plt.axhline(0, color='black', linewidth=1)
plt.tight_layout()


## Benchmark: S&P 500 (optional, if data available)
If a benchmark file is available, we compute a simple monthly equity curve for comparison.
This is used only for context in the thesis, not in the UI.


In [ ]:
BENCH_PATH = Path(OUT_DIR) / 'benchmarks' / 'sp500.parquet'
if BENCH_PATH.exists():
    sp = pd.read_parquet(BENCH_PATH)
    sp['date'] = pd.to_datetime(sp['date'])
    if 'ret_1m' not in sp.columns and 'close' in sp.columns:
        sp = sp.sort_values('date')
        sp['ret_1m'] = sp['close'].pct_change(1)
    sp = sp.dropna(subset=['ret_1m'])
    sp['equity'] = (1 + sp['ret_1m']).cumprod()
    display(sp[['date', 'ret_1m', 'equity']].tail(5))
else:
    print('SP500 benchmark not found. Add data to:', BENCH_PATH)


## Macro regimes
temporal stability, sectors strenght, 

## Fetaure Importance

## PEr sector/ticker
Quali tickers/settori crescono meglio o hanno migliori metriche dipendendo dal modello